# 07 - Restricted-data controls (SPEC section 10)
AUSTRAC data is visible only to members of the account group `austrac_authorised` (configurable widget).
* `is_austrac_authorised()` - the single group check used by every control.
* `secure_austrac_smr` - row-level detail; non-members see the record exists but every field is masked.
* `secure_entity_austrac` - per-entity summary for Entity 360: the **count** is always visible ("Restricted - 3 records", SPEC 7 S5); detail is returned only to authorised users.
* Column masks are applied to AUSTRAC-derived feature columns in `gold_entity_features` where the table type allows it.

The app reads these views **on behalf of the signed-in user**, so Unity Catalog decides what each user sees. AUSTRAC detail is never copied into Lakebase.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("austrac_group", "austrac_authorised")
CATALOG, SCHEMA, GROUP = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema"), dbutils.widgets.get("austrac_group")
assert CATALOG and SCHEMA and GROUP
FQ = f"{CATALOG}.{SCHEMA}"

In [ ]:
spark.sql(f'''CREATE OR REPLACE FUNCTION {FQ}.is_austrac_authorised() RETURNS BOOLEAN
COMMENT 'True when the current user may see AUSTRAC-restricted detail (SPEC 10)'
RETURN is_account_group_member('{GROUP}')''')

spark.sql(f'''CREATE OR REPLACE VIEW {FQ}.secure_austrac_smr
COMMENT 'AUSTRAC SMRs (synthetic, AUSTRAC-RESTRICTED). Detail is masked unless is_austrac_authorised().' AS
SELECT smr_id, acn,
       CASE WHEN {FQ}.is_austrac_authorised() THEN report_date END AS report_date,
       CASE WHEN {FQ}.is_austrac_authorised() THEN smr_type END AS smr_type,
       CASE WHEN {FQ}.is_austrac_authorised() THEN flow_band END AS flow_band,
       classification, provenance, {FQ}.is_austrac_authorised() AS viewer_authorised
FROM {FQ}.silver_austrac_smr''')

spark.sql(f'''CREATE OR REPLACE VIEW {FQ}.secure_entity_austrac
COMMENT 'Per-entity AUSTRAC summary. Count always visible; detail only for authorised users.' AS
SELECT concat('CO-', acn) AS entity_id, acn, count(*) AS restricted_record_count,
       {FQ}.is_austrac_authorised() AS viewer_authorised,
       CASE WHEN {FQ}.is_austrac_authorised() THEN max(report_date) END AS latest_report_date,
       CASE WHEN {FQ}.is_austrac_authorised() THEN sum(CASE WHEN report_date > add_months(current_date(), -24) THEN 1 ELSE 0 END) END AS smr_count_24m,
       CASE WHEN {FQ}.is_austrac_authorised() THEN sum(CASE WHEN flow_band IN ('100k-1m', '>1m') THEN 1 ELSE 0 END) END AS large_flow_count,
       CASE WHEN {FQ}.is_austrac_authorised() THEN array_sort(collect_set(smr_type)) END AS smr_types
FROM {FQ}.silver_austrac_smr GROUP BY acn''')

In [ ]:
# Column masks on AUSTRAC-derived model features. Pipeline-managed tables may reject ALTER; the secure views above
# are the enforced path for the app either way, so report rather than fail.
spark.sql(f'''CREATE OR REPLACE FUNCTION {FQ}.mask_austrac_count(v BIGINT) RETURNS BIGINT
RETURN CASE WHEN {FQ}.is_austrac_authorised() THEN v END''')
for col in ("smr_count_24m", "large_flow_smr_count_24m"):
    try:
        spark.sql(f"ALTER TABLE {FQ}.gold_entity_features ALTER COLUMN {col} SET MASK {FQ}.mask_austrac_count")
        print("masked", col)
    except Exception as e:
        print(f"mask not applied to gold_entity_features.{col}: {str(e).splitlines()[0][:160]}")
display(spark.sql(f"SELECT {FQ}.is_austrac_authorised() AS i_am_authorised, count(*) AS entities FROM {FQ}.secure_entity_austrac"))